# Kaggle Runner: Anatomy of a ReID Attack
**End-to-End Adversarial Robustness Pipeline on Market-1501 (T4 GPU)**

### Hardware & Environment Requirements:
- **Accelerator**: GPU T4 x1
- **Internet**: ON
- **Input Dataset**: `pengcw1/market-1501` (Search and add dataset `Market-1501`)

All scripts feature real-time unbuffered batch logging, per-epoch checkpointing, incremental attack result saving, and `--skip_if_exists` to prevent lost compute.

In [ ]:
# Step 0: Environment Setup & Code Sync
import os, sys, datetime
os.environ["PYTHONUNBUFFERED"] = "1"

!pip -q install timm gradio

if not os.path.exists('reid-attack-anatomy'):
    !git clone https://github.com/udayraj1238/reid-attack-anatomy.git
    %cd reid-attack-anatomy
else:
    %cd reid-attack-anatomy
    !git pull

# Verify environment
!python -u tests/smoke_test.py
print('Setup completed at:', datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


## Step 1 - Train Victim Model Baselines
Trains 3 victim architectures: ResNet-50 (BoT), DenseNet-121, and ViT-Small.
- Batch progress is logged live every 20 batches with elapsed time and ETA.
- Latest checkpoint `{arch}_latest.pth` is saved every epoch to protect progress.
- If a checkpoint already exists, `--skip_if_exists` skips it automatically.

In [ ]:
DATA = '/kaggle/input'
print('\n>>> [Step 1A] Training ResNet-50 Baseline (~20-25 min)...\n')
!python -u -m reidattack.train --data $DATA --arch resnet50 --epochs 30 --log_every 20 --skip_if_exists


In [ ]:
print('\n>>> [Step 1B] Training DenseNet-121 Baseline (~25-30 min)...\n')
!python -u -m reidattack.train --data $DATA --arch densenet121 --epochs 30 --log_every 20 --skip_if_exists


In [ ]:
print('\n>>> [Step 1C] Training ViT-Small Baseline (~35-40 min)...\n')
!python -u -m reidattack.train --data $DATA --arch vit_small --epochs 30 --log_every 20 --skip_if_exists


## Step 2 - Attack Zoo & Cross-Architecture Transfer Matrix
Evaluates FGSM, PGD-10, MI-FGSM, and Logit-PGD across eps in {4, 8}.
- Crafts adversarial queries against surrogate models and evaluates transfer against all targets.
- Saves results incrementally after each configuration so no progress is lost if interrupted.

In [ ]:
CK = 'checkpoints/resnet50.pth checkpoints/densenet121.pth checkpoints/vit_small.pth'
print('\n>>> [Step 2A] Running Evasion Attack Zoo & Transfer Matrix...\n')
!python -u scripts/run_attacks.py --data $DATA --ckpts $CK --eps 4 8 --mode evade

print('\n>>> [Step 2B] Running Impersonation Attacks (PGD, MI-FGSM)...\n')
!python -u scripts/run_attacks.py --data $DATA --ckpts $CK --eps 8 --mode impersonate --attacks pgd mifgsm


## Step 3 - Universal Torso Invisibility Patch (EOT)
Trains a 64x48 universal physical patch on the torso with Expectation Over Transformation (rotation, scale, lighting, noise).

In [ ]:
print('\n>>> [Step 3] Training Universal Physical Patch (5 epochs EOT)...\n')
!python -u scripts/run_patch.py --data $DATA --src checkpoints/resnet50.pth --others checkpoints/densenet121.pth checkpoints/vit_small.pth --epochs 5


## Step 4 - Adversarial Training Defence
Trains ResNet-50 with on-the-fly PGD perturbations (eps=8/255) to quantify the accuracy-robustness trade-off.

In [ ]:
print('\n>>> [Step 4] Running Adversarial Training Defence for ResNet-50...\n')
!python -u -m reidattack.adv_train --data $DATA --arch resnet50 --eps 8 --epochs 20 --log_every 15 --skip_if_exists


## Step 5 - Generate Analysis Visualizations
Generates Baseline Table, Transfer Heatmap, Epsilon Degradation Curves, Grad-CAM Attention Heatmaps, and t-SNE 2D Feature Drift.

In [ ]:
print('\n>>> [Step 5] Generating Publication-Ready Figures in results/...\n')
!python -u scripts/make_baseline_table.py
!python -u scripts/plot_transfer_matrix.py --input results/attacks_evade.json
!python -u scripts/plot_eps_curve.py --input results/attacks_evade.json
!python -u scripts/generate_gradcam.py --ckpt checkpoints/resnet50.pth
!python -u scripts/plot_tsne.py
print('All visualization figures generated successfully!')


## Step 6 - Verification & Artifact Packaging
Validates demo components in non-blocking mode and lists all output files.

In [ ]:
print('\n>>> [Step 6] Validating Gradio UI (Non-blocking Check)...\n')
!python -u demo/app.py --check

print('\n================ SUMMARY OF GENERATED ARTIFACTS ================')
!ls -lh checkpoints/
!ls -lh results/
print('\n>>> ALL PIPELINE STAGES COMPLETED SUCCESSFULLY! <<<')
